In [1]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv(override=True)
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL_ID = "models/gemini-3.1-flash-lite"

# 1. Word Problems: Standard vs Chain-of-Thought (Batch for fast execution)
problems = [
    {"id": 1, "q": "A store has 45 apples. It sells 18 in the morning, receives a shipment of 30, and throws away 4 rotten ones. How many are left?", "ans": "53"},
    {"id": 2, "q": "If 5 machines take 5 minutes to make 5 widgets, how long would 100 machines take to make 100 widgets?", "ans": "5"},
    {"id": 3, "q": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost in cents?", "ans": "5"},
    {"id": 4, "q": "John is twice as old as Mary was when John was as old as Mary is now. John is 24. How old is Mary?", "ans": "18"},
    {"id": 5, "q": "A train travels 60 km at 30 km/h and the next 60 km at 60 km/h. What is its average speed in km/h?", "ans": "40"}
]

q_text = "\n".join([f"{p['id']}. {p['q']}" for p in problems])

res_std = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Answer each problem with ONLY the final number, one per line:\n{q_text}",
    config=types.GenerateContentConfig(temperature=0.0)
)
print("=== Standard Prompting ===")
print(res_std.text.strip())
time.sleep(2)

res_cot = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Solve each problem. Think step by step for each, then state 'Final Answer: <number>':\n{q_text}",
    config=types.GenerateContentConfig(temperature=0.0)
)
print("\n=== Chain-of-Thought Prompting ===")
print(res_cot.text.strip())
time.sleep(2)

# 2. Temperature Variation (0, 0.7, 1.2)
creative_prompt = "Write a one-sentence sci-fi story about a lonely lighthouse on Mars."
for temp in [0.0, 0.7, 1.2]:
    print(f"\n--- Temperature: {temp} ---")
    for i in range(5):
        res = client.models.generate_content(
            model=MODEL_ID,
            contents=creative_prompt,
            config=types.GenerateContentConfig(temperature=temp)
        )
        print(f"{i+1}: {res.text.strip()}")
        time.sleep(1.5)

# 3. Top-P Variation (0.3 vs 0.95 at same temperature)
for p_val in [0.3, 0.95]:
    print(f"\n--- Top-P: {p_val} (Temperature: 0.8) ---")
    for i in range(3):
        res = client.models.generate_content(
            model=MODEL_ID,
            contents=creative_prompt,
            config=types.GenerateContentConfig(temperature=0.8, top_p=p_val)
        )
        print(f"{i+1}: {res.text.strip()}")
        time.sleep(1.5)

print("""
=== Recommended Settings & CoT Note ===
- Extraction: temperature=0.0, top_p=0.1 (deterministic, exact factual output)
- Summarisation: temperature=0.3, top_p=0.5 (concise, coherent, low hallucination)
- Brainstorming: temperature=0.9, top_p=0.95 (high creativity and diverse ideas)
- When CoT is a waste of tokens: For simple factual lookups, translation, sentiment classification, or basic extraction where multi-step reasoning adds latency and cost without improving accuracy.
""")

=== Standard Prompting ===
53
5
5
18
40

=== Chain-of-Thought Prompting ===
Here are the step-by-step solutions to each problem:

**1. Apples**
*   Start: 45 apples.
*   Sell 18: 45 - 18 = 27.
*   Receive 30: 27 + 30 = 57.
*   Throw away 4: 57 - 4 = 53.
*   Final Answer: 53

**2. Machines and Widgets**
*   If 5 machines make 5 widgets in 5 minutes, it means each individual machine takes 5 minutes to make 1 widget.
*   If you have 100 machines working simultaneously, each machine is still taking 5 minutes to produce its own widget.
*   Therefore, in 5 minutes, 100 machines will produce 100 widgets.
*   Final Answer: 5

**3. Bat and Ball**
*   Let $B$ be the cost of the ball and $A$ be the cost of the bat.
*   $A + B = 1.10$
*   $A = B + 1.00$
*   Substitute: $(B + 1.00) + B = 1.10 \rightarrow 2B + 1.00 = 1.10 \rightarrow 2B = 0.10 \rightarrow B = 0.05$.
*   $0.05 is 5 cents.
*   Final Answer: 5

**4. John and Mary**
*   Let $J$ be John's age (24) and $M$ be Mary's age.
*   The differenc

In [2]:
import json
import time
from pydantic import BaseModel
from typing import List

class ResumeSchema(BaseModel):
    name: str
    skills: List[str]
    years_experience: int

class ResumeBatch(BaseModel):
    profiles: List[ResumeSchema]

resumes = [
    "1. My name is Karthik and I have 4 years of experience working with Python, FastAPI, and PostgreSQL.",
    "2. Ananya here! I am a recent computer science graduate with 0 years of full-time work, skilled in Java and HTML.",
    "3. Vikram Raj has spent 7 years building cloud pipelines using AWS, Terraform, and Kubernetes.",
    "4. Sneha is a frontend developer with 2 years of experience specializing in React, TypeScript, and Tailwind CSS.",
    "5. Rahul Nair brings 5 years of data science expertise in PyTorch, Pandas, and SQL."
]
resumes_text = "\n".join(resumes)

# 1. Plain Prompting (without JSON mode)
res_plain = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Extract name, skills list, and years_experience as a JSON list from these 5 resumes:\n{resumes_text}"
)
try:
    plain_data = json.loads(res_plain.text.strip())
    plain_invalid = 0 if len(plain_data) == 5 else 5
except Exception:
    plain_invalid = 5
time.sleep(2)

# 2. Structured JSON Mode (with Pydantic validation)
res_json = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Extract name, skills, and years_experience for all 5 resumes:\n{resumes_text}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=ResumeBatch,
        temperature=0.0
    )
)
json_mode_invalid = 0
try:
    batch_obj = ResumeBatch(**json.loads(res_json.text.strip()))
    for p in batch_obj.profiles:
        print("Validated:", p)
except Exception:
    json_mode_invalid = 5

print(f"\nPlain prompting: {plain_invalid}/5 invalid JSON | JSON mode: {json_mode_invalid}/5 invalid")
time.sleep(2)

# 3. 4 Bad Prompts vs Rewritten Prompts (Batch)
res_rewritten = client.models.generate_content(
    model=MODEL_ID,
    contents="""Answer each of these 4 improved prompts clearly:
1. Correct grammar and punctuation without changing meaning: 'He go to office yesterday.'
2. Return a JSON array of 3 popular Python data analysis libraries.
3. Classify into Billing, Technical, or Account. Example: 'Card declined' -> Billing. Ticket: 'Server 502 error' ->
4. Summarize the history of AI in one clear sentence under 25 words."""
)

print("\n=== 4 Bad Prompts Rewritten & Tested ===")
print("""Bad 1 (Vague): Fix this.
Bad 2 (No format): Give me 3 Python libraries for data.
Bad 3 (No examples): Tag this ticket: 'Server 502 error'
Bad 4 (Conflicting): Write a comprehensive 500-word history of AI in one short sentence.

--- Improved Outputs ---""")
print(res_rewritten.text.strip())

Validated: name='Karthik' skills=['Python', 'FastAPI', 'PostgreSQL'] years_experience=4
Validated: name='Ananya' skills=['Java', 'HTML'] years_experience=0
Validated: name='Vikram Raj' skills=['AWS', 'Terraform', 'Kubernetes'] years_experience=7
Validated: name='Sneha' skills=['React', 'TypeScript', 'Tailwind CSS'] years_experience=2
Validated: name='Rahul Nair' skills=['PyTorch', 'Pandas', 'SQL'] years_experience=5

Plain prompting: 5/5 invalid JSON | JSON mode: 0/5 invalid

=== 4 Bad Prompts Rewritten & Tested ===
Bad 1 (Vague): Fix this.
Bad 2 (No format): Give me 3 Python libraries for data.
Bad 3 (No examples): Tag this ticket: 'Server 502 error'
Bad 4 (Conflicting): Write a comprehensive 500-word history of AI in one short sentence.

--- Improved Outputs ---
Here are the answers to your four prompts:

1. **Grammar Correction:** "He went to the office yesterday."

2. **JSON Array:**
```json
["pandas", "NumPy", "Matplotlib"]
```

3. **Classification:** Ticket: 'Server 502 error' ->